In [5]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
import os


In [11]:
load_dotenv() 
 

True

In [3]:
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=os.getenv("GEMINI_API_KEY"),
    temperature=0.7,
)

In [ ]:
class BlogState(TypedDict):
    title: str
    outline: str
    content: str
    evaluation: str

In [ ]:
def create_outline(state: BlogState) -> BlogState:
    # fetch the title from the state
    prompt = f"Create an outline for a blog post titled '{state['title']}'."
    # generate the outline using the LLM
    outline = model.invoke(prompt)
    # update the state with the generated outline
    state['outline'] = outline
    
    return state
  
  
def create_content(state: BlogState) -> BlogState:
    # fetch the outline from the state
    prompt = f"Create a blog post based on the title - '{state['title']}' using the following outline: {state['outline']}."
    # generate the content using the LLM
    content = model.invoke(prompt)
    # update the state with the generated content
    state['content'] = content
    
    return state

def evaluate_content(state: BlogState) -> BlogState:
    # fetch the content from the state
    prompt = f"Evaluate the following blog post content: {state['content']}. Provide feedback and suggestions for improvement."
    # generate the evaluation using the LLM
    evaluation = model.invoke(prompt)
    # update the state with the generated evaluation
    state['evaluation'] = evaluation
    
    return state

In [ ]:
graph = StateGraph(BlogState)

#add nodes to the graph

graph.add_node("create_outline", create_outline)
graph.add_node("create_content", create_content)
graph.add_node("evaluate_content", evaluate_content)

#add edges to the graph

graph.add_edge(START, "create_outline")
graph.add_edge("create_outline", "create_content")
graph.add_edge("create_content", "evaluate_content")
graph.add_edge("evaluate_content", END)

workflow = graph.compile()

In [9]:
initial_state = {'title': 'The Future of AI in Healthcare'}

final_state = workflow.invoke(initial_state)

print(final_state)

{'title': 'The Future of AI in Healthcare', 'outline': AIMessage(content='Here\'s a comprehensive outline for a blog post titled "The Future of AI in Healthcare," designed to be informative, engaging, and well-structured.\n\n---\n\n## Blog Post Outline: The Future of AI in Healthcare\n\n**Blog Post Title:** The Future of AI in Healthcare: Revolutionizing Wellness, From Diagnosis to Discovery\n\n---\n\n### I. Introduction (Approx. 150-200 words)\n\n*   **A. Catchy Hook:** Start with a compelling statement about the current state of healthcare challenges (e.g., rising costs, physician burnout, limitations in diagnosis) and introduce AI as a powerful disruptor.\n    *   *Example:* "Imagine a healthcare system where diseases are detected years earlier, treatments are perfectly tailored to your unique biology, and medical breakthroughs happen at unprecedented speeds. This isn\'t science fiction; it\'s the promise of Artificial Intelligence in healthcare."\n*   **B. Brief Overview of AI in H

In [10]:
print(final_state['outline'])
print(final_state['content'])

content='Here\'s a comprehensive outline for a blog post titled "The Future of AI in Healthcare," designed to be informative, engaging, and well-structured.\n\n---\n\n## Blog Post Outline: The Future of AI in Healthcare\n\n**Blog Post Title:** The Future of AI in Healthcare: Revolutionizing Wellness, From Diagnosis to Discovery\n\n---\n\n### I. Introduction (Approx. 150-200 words)\n\n*   **A. Catchy Hook:** Start with a compelling statement about the current state of healthcare challenges (e.g., rising costs, physician burnout, limitations in diagnosis) and introduce AI as a powerful disruptor.\n    *   *Example:* "Imagine a healthcare system where diseases are detected years earlier, treatments are perfectly tailored to your unique biology, and medical breakthroughs happen at unprecedented speeds. This isn\'t science fiction; it\'s the promise of Artificial Intelligence in healthcare."\n*   **B. Brief Overview of AI in Healthcare Today:** Briefly touch upon its nascent applications (e